<a href="https://colab.research.google.com/github/OussamaElm0/spark-sql_wc/blob/main/National_Teams_Analytics_Lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!sudo apt update
!apt-get install openjdk-8-jdk-headless -qq > /dev/null
!wget -q https://archive.apache.org/dist/spark/spark-4.1.0/spark-4.1.0-bin-hadoop3.tgz
!tar xf spark-4.1.0-bin-hadoop3.tgz
!pip install -q findspark
!pip install pyspark
!pip install py4j
!pip install -q pymongo matplotlib seaborn

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:10 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,611 kB]
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,868 kB]
Get:13 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,637 kB]
Get:14 https://ppa.lau

In [2]:
import os
import sys
import findspark
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark-4.1.0-bin-hadoop3"
findspark.init()
findspark.find()

'/content/spark-4.1.0-bin-hadoop3'

# **Démarrer une session spark**

In [3]:
import pyspark
from pyspark.sql import SparkSession

spark = SparkSession.builder \
  .appName("Fifa WC") \
  .config("spark.driver.memory", "1G") \
  .getOrCreate()

print("Spark est configuré avec succès!")

Spark est configuré avec succès!


# Lire la dataset

In [4]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


 # **Chargement du Données**

In [5]:
df = spark.read.csv("/content/drive/MyDrive/Colab Notebooks/Py Spark/fifaworldcup.csv", header=True, inferSchema=True)

df.show()

+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|      date|       home_team|away_team|home_score|away_score|tournament|     city| country|neutral|
+----------+----------------+---------+----------+----------+----------+---------+--------+-------+
|1872-11-30|        Scotland|  England|         0|         0|  Friendly|  Glasgow|Scotland|  false|
|1873-03-08|         England| Scotland|         4|         2|  Friendly|   London| England|  false|
|1874-03-07|        Scotland|  England|         2|         1|  Friendly|  Glasgow|Scotland|  false|
|1875-03-06|         England| Scotland|         2|         2|  Friendly|   London| England|  false|
|1876-03-04|        Scotland|  England|         3|         0|  Friendly|  Glasgow|Scotland|  false|
|1876-03-25|        Scotland|    Wales|         4|         0|  Friendly|  Glasgow|Scotland|  false|
|1877-03-03|         England| Scotland|         1|         3|  Friendly|   London| England|  false|


# Nombre de matches présents dans la dataset

In [6]:
df.count()

44152

# La première et la dernière année couverte par les données

In [7]:
first_year = df.select("date").sort("date").first()
print(f"La première année couverte par les données: {first_year.date.year}")

last_year = df.select("date").sort("date", ascending=False).first()
print(f"La dernière année couverte par les données: {last_year.date.year}")

La première année couverte par les données: 1872
La dernière année couverte par les données: 2022


# Les 10 tournois les plus fréquents

In [8]:
df.select("tournament") \
  .groupBy("tournament")\
  .count()\
  .withColumnRenamed("count","matches")\
  .sort("count", ascending=False)\
  .show(10)

+--------------------+-------+
|          tournament|matches|
+--------------------+-------+
|            Friendly|  17461|
|FIFA World Cup qu...|   7774|
|UEFA Euro qualifi...|   2593|
|African Cup of Na...|   1932|
|      FIFA World Cup|    948|
|        Copa América|    841|
|AFC Asian Cup qua...|    764|
|African Cup of Na...|    742|
|          CECAFA Cup|    620|
|CFU Caribbean Cup...|    606|
+--------------------+-------+
only showing top 10 rows


# Nombre des matches ont été joués sur un terrain neutre

In [9]:
df.filter(df.neutral == True).count()

10996

# 10 pays ayant accueilli le plus de matchs

In [10]:
df.select("country")\
  .filter(df.home_team == df.country)\
  .groupBy("country")\
  .count()\
  .sort("count", ascending=False)\
  .show(10)

+-------------+-----+
|      country|count|
+-------------+-----+
|      England|  469|
|       Sweden|  464|
|       France|  447|
|      Hungary|  441|
|      Germany|  433|
|United States|  417|
|      Austria|  408|
|  Switzerland|  401|
|       Norway|  399|
|      Denmark|  388|
+-------------+-----+
only showing top 10 rows


# Nombre de matchs ont terminé sur un score nul

In [11]:
df.filter(df.home_score == df.away_score)\
  .count()

10213

# Matchs où le score total est supérieur à 6

In [12]:
from pyspark.sql.functions import col

df_cleaned = df.filter(
    (df.home_score != "NA") | (df.away_score != "NA")
)

df_cleaned.filter(
    (col("home_score").cast("int") + col("away_score").cast("int")) >= 6
).show(df_cleaned.count())

+----------+--------------------+--------------------+----------+----------+--------------------+--------------------+--------------------+-------+
|      date|           home_team|           away_team|home_score|away_score|          tournament|                city|             country|neutral|
+----------+--------------------+--------------------+----------+----------+--------------------+--------------------+--------------------+-------+
|1873-03-08|             England|            Scotland|         4|         2|            Friendly|              London|             England|  false|
|1878-03-02|            Scotland|             England|         7|         2|            Friendly|             Glasgow|            Scotland|  false|
|1878-03-23|            Scotland|               Wales|         9|         0|            Friendly|             Glasgow|            Scotland|  false|
|1879-04-05|             England|            Scotland|         5|         4|            Friendly|              L

# **Agrégations & statistiques**

# Nombre total de matchs joués par chaque équipe (domicile + extérieur)

In [13]:
home_teams = df.select(col("home_team").alias("team"))
away_teams = df.select(col("away_team").alias("team"))

all_teams = home_teams.union(away_teams)

all_teams.groupBy(col("team"))\
  .count()\
  .sort("count", ascending=False)\
  .show(df.count())

+--------------------+-----+
|                team|count|
+--------------------+-----+
|              Sweden| 1052|
|             England| 1047|
|              Brazil| 1019|
|           Argentina| 1014|
|             Germany|  986|
|             Hungary|  964|
|              Mexico|  926|
|             Uruguay|  919|
|         South Korea|  904|
|              France|  875|
|              Poland|  851|
|               Italy|  837|
|         Switzerland|  834|
|              Norway|  832|
|             Denmark|  832|
|             Austria|  820|
|         Netherlands|  820|
|            Scotland|  814|
|             Belgium|  804|
|               Chile|  793|
|             Finland|  776|
|              Zambia|  766|
|            Paraguay|  748|
|               Spain|  731|
|       United States|  727|
|             Romania|  723|
|              Russia|  716|
|            Bulgaria|  699|
| Trinidad and Tobago|  694|
|               Wales|  692|
|               Japan|  683|
|             

# Top 10 des équipes ayant marqué le plus de buts (toutes compétitions confondues)

In [14]:
cleaned_df = df.filter((col("home_score") != "NA") | (col("away_score") != "NA"))

home_goals = cleaned_df.select(
    col("home_team").alias("team"),
    col("home_score").alias("goals").cast("int")
)
away_goals = cleaned_df.select(
    col("away_team").alias("team"),
    col("away_score").alias("goals").cast("int")
)

total_goals = home_goals.union(away_goals)

total_goals.groupBy("team")\
  .sum("goals")\
  .sort("sum(goals)", ascending=False)\
  .show(10)

+-----------+----------+
|       team|sum(goals)|
+-----------+----------+
|    England|      2282|
|     Brazil|      2228|
|    Germany|      2205|
|     Sweden|      2060|
|    Hungary|      1948|
|  Argentina|      1880|
|Netherlands|      1694|
|     Mexico|      1592|
|South Korea|      1576|
|     France|      1560|
+-----------+----------+
only showing top 10 rows


# Moyenne de buts par match par décennie

In [15]:
from pyspark.sql import functions as F
from pyspark.sql.functions import Column

df = df.withColumn("home_score", Column.try_cast(col("home_score"), "int"))\
       .withColumn("away_score", Column.try_cast(col("away_score"), "int"))

df_filtred = df.filter(col("home_score").isNotNull() & col("away_score").isNotNull())

df_with_goals = df_filtred.withColumn(
    "total_goals",
    col("home_score") + col("away_score")
)

df_with_decade = df_with_goals.withColumn(
    "decade",
    (F.floor(F.year("date") / 10) * 10).cast("int")
)

df_with_decade.groupBy("decade")\
  .avg("total_goals")\
  .sort("decade")\
  .show()

+------+------------------+
|decade|  avg(total_goals)|
+------+------------------+
|  1870| 4.538461538461538|
|  1880| 5.581818181818182|
|  1890|5.1525423728813555|
|  1900| 4.169354838709677|
|  1910| 4.007067137809187|
|  1920|3.8367626886145403|
|  1930|  4.25940594059406|
|  1940| 4.294776119402985|
|  1950| 3.980964467005076|
|  1960|3.4683274021352313|
|  1970| 2.955379908210097|
|  1980|2.4950990615224193|
|  1990| 2.730442692540934|
|  2000| 2.792825302483549|
|  2010|  2.69994863893169|
|  2020| 2.600644864117918|
+------+------------------+



# Nombre de matchs joués par tournoi et par année.

In [16]:
df_with_years = df.withColumn(
    "year",
    F.year("date")
)

df_with_years.groupBy("tournament", "year")\
  .count()\
  .sort("year")\
  .withColumnRenamed("count", "nbr_matches")\
  .show()

+--------------------+----+-----------+
|          tournament|year|nbr_matches|
+--------------------+----+-----------+
|            Friendly|1872|          1|
|            Friendly|1873|          1|
|            Friendly|1874|          1|
|            Friendly|1875|          1|
|            Friendly|1876|          2|
|            Friendly|1877|          2|
|            Friendly|1878|          2|
|            Friendly|1879|          3|
|            Friendly|1880|          3|
|            Friendly|1881|          3|
|            Friendly|1882|          5|
|            Friendly|1883|          5|
|British Championship|1884|          6|
|            Friendly|1885|          1|
|British Championship|1885|          6|
|British Championship|1886|          6|
|            Friendly|1886|          1|
|British Championship|1887|          6|
|            Friendly|1888|          1|
|British Championship|1888|          6|
+--------------------+----+-----------+
only showing top 20 rows


# Classement des équipes ayant remporté le plus de matchs à domicile.

In [17]:
df.filter(
    col("home_score").isNotNull() &
    col("away_score").isNotNull() &
    col("neutral") == False
  )\
  .filter(col("home_score") > col("away_score"))\
  .groupBy("away_team")\
  .count()\
  .sort("count", ascending=False)\
  .show(10)

+----------------+-----+
|       away_team|count|
+----------------+-----+
|         Finland|  242|
|     Switzerland|  194|
|Northern Ireland|  188|
|           Wales|  187|
|          Norway|  187|
|         Hungary|  184|
|          Sweden|  183|
|        Paraguay|  173|
|         Uruguay|  172|
|        Bulgaria|  169|
+----------------+-----+
only showing top 10 rows


# Nombre de victoires, défaites et nuls pour chaque équipe

In [18]:
from pyspark.sql.functions import when

home_results_df = df.filter(
    (col("home_score").isNotNull()) |
    (col("away_score").isNotNull())
).withColumn(
    "team",
    col("home_team")
).withColumn(
    "result",
    when(col("home_score") > col("away_score"), "win")\
      .when(col("home_score") < col("away_score"), "loss")\
      .otherwise("draw")
).select(col("team"), col("result"))

away_result_df = df.filter(
    (col("home_score").isNotNull()) |
    (col("away_score").isNotNull())
).withColumn(
    "team",
    col("away_team")
).withColumn(
    "result",
    when(col("away_score") > col("home_score"), "win")\
      .when(col("away_score") < col("home_score"), "loss")\
      .otherwise("draw")
).select(col("team"), col("result"))

results_df = home_results_df.union(away_result_df)

results_df.groupBy("team")\
  .pivot("result", ["win", "loss", "draw"])\
  .count()\
  .sort("team")\
  .show()

+-------------------+----+----+----+
|               team| win|loss|draw|
+-------------------+----+----+----+
|           Abkhazia|  12|   4|  12|
|        Afghanistan|  32|  61|  28|
|            Albania|  94| 190|  77|
|           Alderney|   3|  16|NULL|
|            Algeria| 238| 160| 149|
|     American Samoa|   5|  41|   2|
|          Andalusia|   8|   1|   4|
|            Andorra|  12| 157|  24|
|             Angola| 130| 118| 133|
|           Anguilla|   3|  58|   7|
|Antigua and Barbuda|  66|  99|  39|
|   Arameans Suryoye|   5|   3|   2|
|          Argentina| 547| 213| 251|
|            Armenia|  59| 128|  50|
|            Artsakh|   6|   3|   2|
|              Aruba|  23|  76|  28|
|           Asturias|   1|NULL|NULL|
|          Australia| 269| 151| 116|
|            Austria| 340| 303| 177|
|             Aymara|NULL|   2|NULL|
+-------------------+----+----+----+
only showing top 20 rows


# Score moyen des matchs joués sur terrain neutre vs non neutre.

In [19]:
from pyspark.sql.functions import avg, count

df.filter(
    (col("home_score").isNotNull()) |
    (col("away_score").isNotNull())
  )\
  .withColumn(
      "total_goals",
      col("home_score") + col("away_score")
  )\
  .select(
      col("neutral"),
      col("total_goals")
  )\
  .groupBy("neutral")\
  .agg(
      count("*").alias("total_matches"), avg("total_goals").alias("avg_goals")
  )\
  .show()

+-------+-------------+------------------+
|neutral|total_matches|         avg_goals|
+-------+-------------+------------------+
|   true|        10951|3.0150671171582504|
|  false|        33153|2.8861339848580823|
+-------+-------------+------------------+



# Top 5 des matchs avec l’écart de score le plus élevé

In [20]:
from pyspark.sql.functions import abs

df.fillna(0)\
  .withColumn(
      "score_difference",
      abs(col("home_score") - col("away_score"))
  )\
  .sort("score_difference", ascending=False)\
  .show(5)

+----------+---------+--------------+----------+----------+--------------------+-------------+----------------+-------+----------------+
|      date|home_team|     away_team|home_score|away_score|          tournament|         city|         country|neutral|score_difference|
+----------+---------+--------------+----------+----------+--------------------+-------------+----------------+-------+----------------+
|2001-04-11|Australia|American Samoa|        31|         0|FIFA World Cup qu...|Coffs Harbour|       Australia|  false|              31|
|1971-09-13|   Tahiti|  Cook Islands|        30|         0| South Pacific Games|      Papeete|French Polynesia|  false|              30|
|1979-08-30|     Fiji|      Kiribati|        24|         0| South Pacific Games|      Nausori|            Fiji|  false|              24|
|2001-04-09|Australia|         Tonga|        22|         0|FIFA World Cup qu...|Coffs Harbour|       Australia|  false|              22|
|1966-04-03|    Libya|          Oman|    

# **Requêtes analytiques (fenêtres & logique avancée)**

# Calculer le goal average (buts marqués - buts encaissés) par équipe

In [21]:
home_goals_df = df.fillna(0)\
  .withColumn(
      "team",
      col("home_team")
  )\
  .withColumn(
      "goals_for",
      col("home_score")
  )\
  .withColumn(
      "goals_against",
      col("away_score")
  )\
  .select(
      col("team"),
      col("goals_for"),
      col("goals_against")
  )

away_goals_df = df.fillna(0)\
  .withColumn(
      "team",
      col("away_team")
  )\
  .withColumn(
      "goals_for",
      col("away_score")
  )\
  .withColumn(
      "goals_against",
      col("home_score")
  )\
  .select(
      col("team"),
      col("goals_for"),
      col("goals_against")
  )

teams_with_goals_df = home_goals_df.union(away_goals_df)

teams_with_goals_df.groupBy("team")\
  .sum("goals_for", "goals_against")\
  .withColumnRenamed(
      "sum(goals_for)",
      "goals_for"
  )\
  .withColumnRenamed(
      "sum(goals_against)",
      "goals_against"
  )\
  .withColumn(
      "goal_avg",
      col("goals_for") - col("goals_against")
  )\
  .sort("goal_avg", ascending=False)\
  .show()

+-----------+---------+-------------+--------+
|       team|goals_for|goals_against|goal_avg|
+-----------+---------+-------------+--------+
|     Brazil|     2228|          910|    1318|
|    England|     2282|         1014|    1268|
|    Germany|     2205|         1133|    1072|
|  Argentina|     1880|         1042|     838|
|      Spain|     1464|          650|     814|
|South Korea|     1576|          808|     768|
|     Sweden|     2060|         1347|     713|
|Netherlands|     1694|         1000|     694|
|      Italy|     1441|          803|     638|
|     Mexico|     1592|          976|     616|
|       Iran|      968|          412|     556|
|     Russia|     1229|          696|     533|
|    Hungary|     1948|         1429|     519|
|  Australia|     1081|          583|     498|
|     France|     1560|         1128|     432|
|   China PR|     1097|          674|     423|
|      Egypt|     1096|          680|     416|
|      Japan|     1189|          785|     404|
|   Scotland|

# Classement des équipes par nombre de victoires par année

In [22]:
from pyspark.sql.functions import year, row_number
from pyspark.sql.window import Window

# Nombre de victoires par année
matches_with_winner_year_df = df.fillna(0)\
  .withColumn(
      "winner",
      when(col("home_score") > col("away_score"), col("home_team"))\
        .when(col("home_score") < col("away_score"), col("away_team"))\
          .otherwise(None)
  )\
  .withColumn(
      "year",
      year(col("date"))
  )\
  .select(
      col("winner"),
      col("year")
  )

wins_per_year_df = matches_with_winner_year_df.filter(
      col("winner").isNotNull()
    )\
    .withColumnRenamed(
        "winner",
        "team"
    )\
    .groupBy("team", "year")\
    .count()

# Classement des équipes
window_spec = Window.partitionBy("year").orderBy(col("count").desc())
ranking_df = wins_per_year_df.withColumn(
    "ranking",
    row_number().over(window_spec)
)\
.withColumnRenamed(
    "count",
    "wins"
)\
.show()

+--------+----+----+-------+
|    team|year|wins|ranking|
+--------+----+----+-------+
| England|1873|   1|      1|
|Scotland|1874|   1|      1|
|Scotland|1876|   2|      1|
|Scotland|1877|   2|      1|
|Scotland|1878|   2|      1|
| England|1879|   2|      1|
|Scotland|1879|   1|      2|
|Scotland|1880|   2|      1|
| England|1880|   1|      2|
|Scotland|1881|   2|      1|
|   Wales|1881|   1|      2|
|   Wales|1882|   2|      1|
|Scotland|1882|   2|      2|
| England|1882|   1|      3|
|Scotland|1883|   2|      1|
| England|1883|   2|      2|
|Scotland|1884|   3|      1|
| England|1884|   2|      2|
|   Wales|1884|   1|      3|
|Scotland|1885|   2|      1|
+--------+----+----+-------+
only showing top 20 rows


# Évolution du nombre de matchs par décennie

In [23]:
from pyspark.sql.functions import floor, year

matches_per_decade_df = df.withColumn(
    "decade",
     (floor(year(col("date")) / 10) * 10)
  )\
  .select(col("decade"))


matches_per_decade_df.groupBy("decade")\
  .count()\
  .sort("count")\
  .withColumnRenamed(
      "count",
      "total_matches"
  )\
  .show()

+------+-------------+
|decade|total_matches|
+------+-------------+
|  1870|           13|
|  1880|           55|
|  1890|           59|
|  1900|          124|
|  1910|          283|
|  1920|          729|
|  1940|          804|
|  1930|         1010|
|  1950|         1576|
|  2020|         2219|
|  1960|         2810|
|  1970|         3922|
|  1980|         4795|
|  1990|         6596|
|  2000|         9422|
|  2010|         9735|
+------+-------------+



# Identifier les équipes invaincues sur une année donnée

In [24]:
from pyspark.sql.functions import when, year

home_matches_results_df = df.fillna(0)\
  .withColumn(
      "team",
      col("home_team")
  )\
  .withColumn(
      "result",
      when(col("home_score") > col("away_score"), "win")\
        .when(col("home_score") < col("away_score"), "loss")\
          .otherwise("draw")
  )\
  .withColumn(
      "year",
      year(col("date"))
  )\
  .select(
      col("team"),
      col("result"),
      col("year")
  )

away_matches_results_df = df.fillna(0)\
  .withColumn(
      "team",
      col("away_team")
  )\
  .withColumn(
      "result",
      when(col("away_score") > col("home_score"), "win")\
        .when(col("away_score") < col("home_score"), "loss")\
          .otherwise("draw")
  )\
  .withColumn(
      "year",
      year(col("date"))
  )\
  .select(
      col("team"),
      col("result"),
      col("year")
  )

teams_losses_per_year = home_matches_results_df.union(away_matches_results_df)\
  .groupBy("team", "year")\
  .pivot("result", ["win", "loss", "draw"])\
  .count()\
  .fillna(0)\
  .sort("year")\
  .select(
      col("team"),
      col("year"),
      col("loss").cast("int")
  )

print("Enter a year: ")
input_year = int(input())

print(f"Undefeated teams over {input_year}")
teams_losses_per_year.filter(
    (col("year") == input_year) &
    (col("loss") == 0)
  )\
  .show()

Enter a year: 
1990
Undefeated teams over 1990
+--------------------+----+----+
|                team|year|loss|
+--------------------+----+----+
|             Réunion|1990|   0|
|             Vanuatu|1990|   0|
|           Andalusia|1990|   0|
|        Sint Maarten|1990|   0|
|      Basque Country|1990|   0|
|       Guinea-Bissau|1990|   0|
|               Chile|1990|   0|
|               Italy|1990|   0|
|          Madagascar|1990|   0|
|                Fiji|1990|   0|
|        Sierra Leone|1990|   0|
|              Canada|1990|   0|
|             Georgia|1990|   0|
|             Curaçao|1990|   0|
|         New Zealand|1990|   0|
|            Thailand|1990|   0|
|Saint Vincent and...|1990|   0|
|             Algeria|1990|   0|
|               Yemen|1990|   0|
|            Portugal|1990|   0|
+--------------------+----+----+
only showing top 20 rows


# La longue série de victoires consécutives par équipe

In [25]:
from pyspark.sql.functions import when, sum as sparksum, max as sparkmax
from pyspark.sql.window import Window

home_per_matche_df = df.fillna(0)\
  .withColumn(
      "team",
      col("home_team")
  )\
  .withColumn(
      "is_winner",
      when(col("home_score") > col("away_score"), 1)\
        .otherwise(0)
  )\
  .select(
      col("team"),
      col("is_winner"),
      col("date")
  )

away_per_matche_df = df.fillna(0)\
  .withColumn(
      "team",
      col("away_team")
  )\
  .withColumn(
      "is_winner",
      when(col("away_score") > col("home_score"), 1)\
        .otherwise(0)
  )\
  .select(
      col("team"),
      col("is_winner"),
      col("date")
  )

window_spec = Window.partitionBy("team").orderBy("date")
teams_per_match_df = home_per_matche_df.union(away_per_matche_df)\
  .withColumn(
      "streak_group",
      sparksum(when(col("is_winner") == 0, 1).otherwise(0)).over(window_spec)
  )

wins_df = teams_per_match_df.filter(col("is_winner") == 1)

streaks_df = wins_df.groupBy("team", "streak_group") \
    .agg(count("*").alias("win_streak"))

longest_streak = streaks_df.groupBy("team") \
    .agg(sparkmax("win_streak").alias("longest_win_streak"))

longest_streak.sort("longest_win_streak", ascending=False)\
  .show()


+--------------------+------------------+
|                team|longest_win_streak|
+--------------------+------------------+
|           Mauritius|                17|
|             Padania|                15|
|               Spain|                15|
|              France|                14|
|              Brazil|                14|
|           Australia|                13|
|              Guyana|                13|
|               Italy|                13|
|              Mexico|                13|
|            Scotland|                13|
|             Belgium|                12|
|           German DR|                12|
|             Germany|                12|
|           Indonesia|                12|
|             Morocco|                12|
|              Russia|                12|
|        Saudi Arabia|                11|
|              Sweden|                11|
|United Arab Emirates|                11|
|       United States|                11|
+--------------------+------------

# L'équipe la plus victorieuse pour chaque tournoi

In [26]:
from pyspark.sql.functions import when
from pyspark.sql.window import Window

df_with_tournaments_winners = df.fillna(0)\
  .withColumn(
      "winner",
      when(col("home_score") > col("away_score"), col("home_team"))\
        .when(col("home_score") < col("away_score"), col("away_team"))\
          .otherwise(None)
  )\
  .select(
      col("winner"),
      col("tournament")
  )\
  .filter(col("winner").isNotNull())\
  .groupBy("winner", "tournament")\
  .count()\
  .withColumnRenamed(
      "count",
      "nb_wins"
  )

window_spec = Window.partitionBy("tournament").orderBy(col("nb_wins").desc())

tournament_wins_ranking_df = df_with_tournaments_winners.withColumn(
    "ranking",
    row_number().over(window_spec)
  )\
  .filter(col("ranking") == 1)

tournament_wins_ranking_df.sort("tournament")\
  .show(tournament_wins_ranking_df.count())

+--------------------+--------------------+-------+-------+
|              winner|          tournament|nb_wins|ranking|
+--------------------+--------------------+-------+-------+
|            Suriname|     ABCS Tournament|      7|      1|
|                Iran|       AFC Asian Cup|     41|      1|
|                Iran|AFC Asian Cup qua...|     36|      1|
|         North Korea|   AFC Challenge Cup|     11|      1|
|         Philippines|AFC Challenge Cup...|      7|      1|
|            Thailand|    AFF Championship|     42|      1|
|              Brunei|AFF Championship ...|      2|      1|
|               Egypt|African Cup of Na...|     60|      1|
|         Ivory Coast|African Cup of Na...|     67|      1|
|             Morocco|African Nations C...|     11|      1|
|              Uganda|African Nations C...|      4|      1|
|               India|    Afro-Asian Games|      2|      1|
|             Senegal|  Amílcar Cabral Cup|     38|      1|
|                Iraq|            Arab C

# les performances à domicile vs à l’extérieur pour chaque équipe

In [43]:
from pyspark.sql.functions import when, lit

home_performance_df = df.fillna(0)\
  .withColumn(
      "team",
      col("home_team")
  )\
  .withColumn(
      "is_home_team",
      lit(True)
  )\
  .withColumn(
      "result",
      when(col("home_score") > col("away_score"), "win")\
        .when(col("home_score") < col("away_score"), "loss")\
          .otherwise("draw")
  )\
  .withColumn(
      "goals_for",
      col("home_score")
  )\
  .withColumn(
      "goals_against",
      col("away_score")
  )\
  .select(
      col("team"),
      col("tournament"),
      col("is_home_team"),
      col("result"),
      col("goals_for"),
      col("goals_against"),
      col("neutral"),
  )

away_performance_df = df.fillna(0)\
  .withColumn(
      "team",
      col("away_team")
  )\
  .withColumn(
      "is_home_team",
      lit(False)
  )\
  .withColumn(
      "result",
      when(col("away_score") > col("home_score"), "win")\
        .when(col("away_score") < col("home_score"), "loss")\
          .otherwise("draw")
  )\
  .withColumn(
      "goals_for",
      col("away_score")
  )\
  .withColumn(
      "goals_against",
      col("home_score")
  )\
  .select(
      col("team"),
      col("tournament"),
      col("is_home_team"),
      col("result"),
      col("goals_for"),
      col("goals_against"),
      col("neutral"),
  )

teams_performance_df = home_performance_df.union(away_performance_df)\
  .withColumn(
      "venue",
      when(col("is_home_team") == True, "home")\
        .otherwise("away")
  )\
  .select(
      col("team"),
      col("tournament"),
      col("venue"),
      col("neutral"),
      col("result")
  )

teams_performance_df.groupBy("team", "tournament", "venue", "neutral")\
  .pivot("result", ["win", "loss", "draw"])\
  .count()\
  .fillna(0)\
  .sort("team", "tournament")\
  .show()

+-----------+--------------------+-----+-------+---+----+----+
|       team|          tournament|venue|neutral|win|loss|draw|
+-----------+--------------------+-----+-------+---+----+----+
|   Abkhazia|CONIFA European F...| away|   true|  1|   1|   2|
|   Abkhazia|CONIFA European F...| home|   true|  2|   0|   2|
|   Abkhazia|CONIFA European F...| away|  false|  0|   0|   2|
|   Abkhazia|CONIFA World Foot...| home|   true|  3|   2|   3|
|   Abkhazia|CONIFA World Foot...| away|  false|  1|   0|   0|
|   Abkhazia|CONIFA World Foot...| home|  false|  4|   0|   1|
|   Abkhazia|CONIFA World Foot...| away|   true|  1|   0|   1|
|   Abkhazia|            Friendly| away|  false|  0|   1|   0|
|   Abkhazia|            Friendly| home|  false|  0|   0|   1|
|Afghanistan|AFC Asian Cup qua...| home|   true|  1|   0|   3|
|Afghanistan|AFC Asian Cup qua...| away|  false|  0|   8|   1|
|Afghanistan|AFC Asian Cup qua...| away|   true|  1|   6|   2|
|Afghanistan|   AFC Challenge Cup| home|   true|  1|   